In [2]:
!pip install -q kaggle

In [1]:
from google.colab import files

files.upload()

Saving kaggle (2).json to kaggle (2).json


{'kaggle (2).json': b'{"username":"ak12an","key":"3eac5c3f4c1edba9f87fbf5a80b0a343"}'}

In [3]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)

!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

print("Kaggle API setup successful!")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Kaggle API setup successful!


In [4]:
!kaggle datasets download -d mohitsingh1804/plantvillage

Dataset URL: https://www.kaggle.com/datasets/mohitsingh1804/plantvillage
License(s): GPL-2.0
100% 818M/818M [00:06<00:00, 133MB/s]



In [5]:
!ls -lh /content

total 818M
-rw-r--r-- 1 root root   62 Sep  4 04:45 'kaggle (2).json'
-rw-r--r-- 1 root root 818M Aug 20  2021  plantvillage.zip
drwxr-xr-x 1 root root 4.0K Aug 24 13:21  sample_data


In [6]:
import zipfile
import os

ZIP_PATH = "/content/plantvillage.zip"
EXTRACT_PATH = "/content/PlantVillage"

# Extract folder तयार करा
os.makedirs(EXTRACT_PATH, exist_ok=True)

print("Extracting dataset... Please wait.")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("✅ Dataset extracted successfully!")

Extracting dataset... Please wait.
✅ Dataset extracted successfully!


In [7]:
import os

DATASET_PATH = "/content/PlantVillage"

print("Folders inside PlantVillage:")
print(os.listdir(DATASET_PATH))

Folders inside PlantVillage:
['PlantVillage']


In [8]:
import os

DATASET_PATH = "/content/PlantVillage/PlantVillage"

print("First folders/classes:")
print(os.listdir(DATASET_PATH)[:10])

print("\nTotal folders:", len(os.listdir(DATASET_PATH)))

First folders/classes:
['train', 'val']

Total folders: 2


In [9]:
import os

TRAIN_DIR = "/content/PlantVillage/PlantVillage/train"
VAL_DIR = "/content/PlantVillage/PlantVillage/val"

train_classes = os.listdir(TRAIN_DIR)
val_classes = os.listdir(VAL_DIR)

print("Number of Training Classes:", len(train_classes))
print("Number of Validation Classes:", len(val_classes))

print("\nFirst 10 Training Classes:")
print(train_classes[:10])

Number of Training Classes: 38
Number of Validation Classes: 38

First 10 Training Classes:
['Apple___Black_rot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Grape___Esca_(Black_Measles)', 'Tomato___Early_blight', 'Strawberry___Leaf_scorch', 'Squash___Powdery_mildew', 'Corn_(maize)___Common_rust_', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Late_blight', 'Soybean___healthy']


In [10]:
import os

TRAIN_DIR = "/content/PlantVillage/PlantVillage/train"
VAL_DIR = "/content/PlantVillage/PlantVillage/val"

# Training images count
train_images = 0

for class_name in os.listdir(TRAIN_DIR):
    class_path = os.path.join(TRAIN_DIR, class_name)

    if os.path.isdir(class_path):
        train_images += len([
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])


# Validation images count
val_images = 0

for class_name in os.listdir(VAL_DIR):
    class_path = os.path.join(VAL_DIR, class_name)

    if os.path.isdir(class_path):
        val_images += len([
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])


print("Total Training Images:", train_images)
print("Total Validation Images:", val_images)
print("Total Images:", train_images + val_images)

Total Training Images: 43444
Total Validation Images: 10861
Total Images: 54305


In [11]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load Training Dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Load Validation Dataset
val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Class Names
class_names = train_dataset.class_names

print("\nDataset Loaded Successfully! ✅")
print("Number of Classes:", len(class_names))

print("\nFirst 10 Classes:")
for i, class_name in enumerate(class_names[:10]):
    print(i, "-", class_name)

Found 43444 files belonging to 38 classes.
Found 10861 files belonging to 38 classes.

Dataset Loaded Successfully! ✅
Number of Classes: 38

First 10 Classes:
0 - Apple___Apple_scab
1 - Apple___Black_rot
2 - Apple___Cedar_apple_rust
3 - Apple___healthy
4 - Blueberry___healthy
5 - Cherry_(including_sour)___Powdery_mildew
6 - Cherry_(including_sour)___healthy
7 - Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
8 - Corn_(maize)___Common_rust_
9 - Corn_(maize)___Northern_Leaf_Blight


In [12]:
AUTOTUNE = tf.data.AUTOTUNE

# Improve performance
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

print("Dataset optimization completed! ✅")

Dataset optimization completed! ✅


In [13]:
from tensorflow.keras import layers, models

NUM_CLASSES = 38

model = models.Sequential([

    # Input Layer
    layers.Input(shape=(224, 224, 3)),

    # Normalize image pixels
    layers.Rescaling(1./255),

    # CNN Block 1
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # CNN Block 2
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # CNN Block 3
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # CNN Block 4
    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Flatten
    layers.Flatten(),

    # Dense Layers
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.5),

    # Output Layer
    layers.Dense(NUM_CLASSES, activation="softmax")
])

# Compile Model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("CNN Model Created Successfully! ✅")

model.summary()

CNN Model Created Successfully! ✅


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │        19,494 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,282,790 (73.56 MB)

 Trainable params: 19,282,790 (73.56 MB)

 Non-trainable params: 0 (0.00 B)

In [1]:
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)

NameError: name 'model' is not defined

In [2]:
print("train_dataset:", "train_dataset" in globals())
print("val_dataset:", "val_dataset" in globals())
print("model:", "model" in globals())

train_dataset: False
val_dataset: False
model: False


In [3]:
from tensorflow.keras import layers, models

NUM_CLASSES = 38

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(512, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("✅ Model created successfully!")

✅ Model created successfully!


In [5]:
import os

print("Train folder exists:", os.path.exists("/content/PlantVillage/PlantVillage/train"))
print("Validation folder exists:", os.path.exists("/content/PlantVillage/PlantVillage/val"))

Train folder exists: False
Validation folder exists: False


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
SOURCE_DIR = "/content/drive/MyDrive/PlantVillage-Dataset.zip"
DEST_DIR = "/content/drive/MyDrive/PlantVillage-Dataset_small.zip"

In [9]:
import os

base = "/content/drive/MyDrive"

print("PlantVillage related files/folders:")

for item in os.listdir(base):
    if "plant" in item.lower():
        path = os.path.join(base, item)
        print(item, "→", "Folder" if os.path.isdir(path) else "File")

PlantVillage related files/folders:
PlantVillage-Dataset.zip → File
PlantVillage_small → Folder


In [10]:
import os
import zipfile

SOURCE_DIR = "/content/drive/MyDrive/PlantVillage-Dataset.zip"

print("Exists:", os.path.exists(SOURCE_DIR))
print("Size:", round(os.path.getsize(SOURCE_DIR) / (1024**3), 2), "GB")

try:
    with zipfile.ZipFile(SOURCE_DIR, 'r') as z:
        print("ZIP is valid ✅")
        print("Number of files:", len(z.namelist()))
        print("First 10 files:")
        for f in z.namelist()[:10]:
            print(f)
except zipfile.BadZipFile:
    print("❌ ZIP file is corrupted or incomplete")
except Exception as e:
    print("❌ Error:", e)

Exists: True
Size: 0.8 GB
ZIP is valid ✅
Number of files: 54305
First 10 files:
PlantVillage/train/Apple___Apple_scab/01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Scab 3003.JPG
PlantVillage/train/Apple___Apple_scab/01f3deaa-6143-4b6c-9c22-620a46d8be04___FREC_Scab 3112.JPG
PlantVillage/train/Apple___Apple_scab/0208f4eb-45a4-4399-904e-989ac2c6257c___FREC_Scab 3037.JPG
PlantVillage/train/Apple___Apple_scab/023123cb-7b69-4c9f-a521-766d7c8543bb___FREC_Scab 3487.JPG
PlantVillage/train/Apple___Apple_scab/0261a6e4-21f8-481a-8827-b674e6955644___FREC_Scab 3055.JPG
PlantVillage/train/Apple___Apple_scab/029424b0-0ef5-491b-9ef5-069190d24d8f___FREC_Scab 3504.JPG
PlantVillage/train/Apple___Apple_scab/0340dc35-5215-48ab-8db7-06af99fcb358___FREC_Scab 2966.JPG
PlantVillage/train/Apple___Apple_scab/03eccb1a-0368-4ac7-9f48-7546037b775a___FREC_Scab 3334.JPG
PlantVillage/train/Apple___Apple_scab/0537ad56-3d24-4e97-a947-65b9b37f8988___FREC_Scab 3079.JPG
PlantVillage/train/Apple___Apple_scab/058d5e64-2c57-45ba

In [11]:
import os
import zipfile
import shutil
from collections import defaultdict

SOURCE_ZIP = "/content/drive/MyDrive/PlantVillage-Dataset.zip"
DEST_DIR = "/content/drive/MyDrive/PlantVillage_small"

IMAGES_PER_CLASS = 500

# Create destination folder
os.makedirs(DEST_DIR, exist_ok=True)

print("Reading ZIP...")

with zipfile.ZipFile(SOURCE_ZIP, "r") as z:

    # Get image files from train directory
    image_files = [
        f for f in z.namelist()
        if f.startswith("PlantVillage/train/")
        and f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print("Total images found:", len(image_files))

    # Group images by class
    class_images = defaultdict(list)

    for file_path in image_files:
        parts = file_path.split("/")

        # Expected:
        # PlantVillage/train/ClassName/image.jpg
        if len(parts) >= 4:
            class_name = parts[2]
            class_images[class_name].append(file_path)

    print("Total classes:", len(class_images))

    # Copy selected images
    total_copied = 0

    for class_name, files in sorted(class_images.items()):

        selected_files = files[:IMAGES_PER_CLASS]

        class_dest = os.path.join(DEST_DIR, class_name)
        os.makedirs(class_dest, exist_ok=True)

        print(f"{class_name}: {len(selected_files)} images")

        for file_path in selected_files:

            filename = os.path.basename(file_path)

            destination = os.path.join(class_dest, filename)

            with z.open(file_path) as source:
                with open(destination, "wb") as target:
                    shutil.copyfileobj(source, target)

            total_copied += 1

print("\n================================")
print("Dataset reduction completed ✅")
print("Total images copied:", total_copied)
print("Output folder:", DEST_DIR)
print("================================")

Reading ZIP...
Total images found: 43444
Total classes: 38
Apple___Apple_scab: 500 images
Apple___Black_rot: 496 images
Apple___Cedar_apple_rust: 220 images
Apple___healthy: 500 images
Blueberry___healthy: 500 images
Cherry_(including_sour)___Powdery_mildew: 500 images
Cherry_(including_sour)___healthy: 500 images
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 410 images
Corn_(maize)___Common_rust_: 500 images
Corn_(maize)___Northern_Leaf_Blight: 500 images
Corn_(maize)___healthy: 500 images
Grape___Black_rot: 500 images
Grape___Esca_(Black_Measles): 500 images
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 500 images
Grape___healthy: 339 images
Orange___Haunglongbing_(Citrus_greening): 500 images
Peach___Bacterial_spot: 500 images
Peach___healthy: 288 images
Pepper,_bell___Bacterial_spot: 500 images
Pepper,_bell___healthy: 500 images
Potato___Early_blight: 500 images
Potato___Late_blight: 500 images
Potato___healthy: 121 images
Raspberry___healthy: 297 images
Soybean___healthy: 500 

In [12]:
import os

DATASET_DIR = "/content/drive/MyDrive/PlantVillage_small"

classes = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])

print("Number of classes:", len(classes))
print("\nClasses:")

for i, cls in enumerate(classes):
    count = len(os.listdir(os.path.join(DATASET_DIR, cls)))
    print(f"{i}: {cls} -> {count} images")

Number of classes: 38

Classes:
0: Apple___Apple_scab -> 500 images
1: Apple___Black_rot -> 496 images
2: Apple___Cedar_apple_rust -> 220 images
3: Apple___healthy -> 500 images
4: Blueberry___healthy -> 500 images
5: Cherry_(including_sour)___Powdery_mildew -> 500 images
6: Cherry_(including_sour)___healthy -> 500 images
7: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot -> 410 images
8: Corn_(maize)___Common_rust_ -> 500 images
9: Corn_(maize)___Northern_Leaf_Blight -> 500 images
10: Corn_(maize)___healthy -> 500 images
11: Grape___Black_rot -> 500 images
12: Grape___Esca_(Black_Measles) -> 500 images
13: Grape___Leaf_blight_(Isariopsis_Leaf_Spot) -> 500 images
14: Grape___healthy -> 339 images
15: Orange___Haunglongbing_(Citrus_greening) -> 500 images
16: Peach___Bacterial_spot -> 500 images
17: Peach___healthy -> 288 images
18: Pepper,_bell___Bacterial_spot -> 500 images
19: Pepper,_bell___healthy -> 500 images
20: Potato___Early_blight -> 500 images
21: Potato___Late_blight -> 

In [13]:
import os
import shutil
import random

# ==============================
# PATHS
# ==============================

SOURCE_DIR = "/content/drive/MyDrive/PlantVillage_small"
OUTPUT_DIR = "/content/drive/MyDrive/PlantVillage_Split"

# ==============================
# SETTINGS
# ==============================

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

SEED = 42
random.seed(SEED)

# ==============================
# CHECK SOURCE
# ==============================

if not os.path.exists(SOURCE_DIR):
    raise FileNotFoundError(
        f"Dataset folder not found: {SOURCE_DIR}"
    )

print("Source dataset found ✅")

# ==============================
# CREATE OUTPUT FOLDERS
# ==============================

for split in ["train", "validation", "test"]:
    os.makedirs(
        os.path.join(OUTPUT_DIR, split),
        exist_ok=True
    )

# ==============================
# GET CLASSES
# ==============================

classes = sorted([
    d for d in os.listdir(SOURCE_DIR)
    if os.path.isdir(os.path.join(SOURCE_DIR, d))
])

print("Number of classes:", len(classes))

# ==============================
# SPLIT DATA
# ==============================

total_images = 0

for class_name in classes:

    source_class_dir = os.path.join(
        SOURCE_DIR,
        class_name
    )

    # Only image files
    images = [
        f for f in os.listdir(source_class_dir)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp")
        )
    ]

    # Shuffle
    random.shuffle(images)

    total = len(images)

    # Calculate split sizes
    train_count = int(total * TRAIN_RATIO)
    val_count = int(total * VAL_RATIO)

    train_images = images[:train_count]

    val_images = images[
        train_count:
        train_count + val_count
    ]

    test_images = images[
        train_count + val_count:
    ]

    # Create class folders
    train_dir = os.path.join(
        OUTPUT_DIR,
        "train",
        class_name
    )

    val_dir = os.path.join(
        OUTPUT_DIR,
        "validation",
        class_name
    )

    test_dir = os.path.join(
        OUTPUT_DIR,
        "test",
        class_name
    )

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # Copy files
    for image in train_images:
        shutil.copy2(
            os.path.join(source_class_dir, image),
            os.path.join(train_dir, image)
        )

    for image in val_images:
        shutil.copy2(
            os.path.join(source_class_dir, image),
            os.path.join(val_dir, image)
        )

    for image in test_images:
        shutil.copy2(
            os.path.join(source_class_dir, image),
            os.path.join(test_dir, image)
        )

    total_images += total

    print(
        f"{class_name}: "
        f"Total={total}, "
        f"Train={len(train_images)}, "
        f"Val={len(val_images)}, "
        f"Test={len(test_images)}"
    )

# ==============================
# FINAL RESULT
# ==============================

print("\n====================================")
print("DATASET SPLIT COMPLETED ✅")
print("====================================")

print("Classes:", len(classes))
print("Total images:", total_images)

print("\nOutput:")
print(OUTPUT_DIR)

Source dataset found ✅
Number of classes: 38
Apple___Apple_scab: Total=500, Train=350, Val=75, Test=75
Apple___Black_rot: Total=496, Train=347, Val=74, Test=75
Apple___Cedar_apple_rust: Total=220, Train=154, Val=33, Test=33
Apple___healthy: Total=500, Train=350, Val=75, Test=75
Blueberry___healthy: Total=500, Train=350, Val=75, Test=75
Cherry_(including_sour)___Powdery_mildew: Total=500, Train=350, Val=75, Test=75
Cherry_(including_sour)___healthy: Total=500, Train=350, Val=75, Test=75
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: Total=410, Train=287, Val=61, Test=62
Corn_(maize)___Common_rust_: Total=500, Train=350, Val=75, Test=75
Corn_(maize)___Northern_Leaf_Blight: Total=500, Train=350, Val=75, Test=75
Corn_(maize)___healthy: Total=500, Train=350, Val=75, Test=75
Grape___Black_rot: Total=500, Train=350, Val=75, Test=75
Grape___Esca_(Black_Measles): Total=500, Train=350, Val=75, Test=75
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): Total=500, Train=350, Val=75, Test=75
Grape___

In [14]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/PlantVillage_Split"

for split in ["train", "validation", "test"]:

    split_dir = os.path.join(OUTPUT_DIR, split)

    total = 0
    class_count = 0

    for class_name in os.listdir(split_dir):

        class_dir = os.path.join(split_dir, class_name)

        if os.path.isdir(class_dir):

            count = len([
                f for f in os.listdir(class_dir)
                if f.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp")
                )
            ])

            total += count
            class_count += 1

    print(
        f"{split.upper()}: "
        f"{total} images | "
        f"{class_count} classes"
    )

TRAIN: 12130 images | 38 classes
VALIDATION: 2596 images | 38 classes
TEST: 2608 images | 38 classes


In [15]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [16]:
import os
import json
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# ============================================================
# 1. PATHS
# ============================================================

DATASET_DIR = "/content/drive/MyDrive/PlantVillage_Split"

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "validation")
TEST_DIR = os.path.join(DATASET_DIR, "test")

MODEL_DIR = "/content/drive/MyDrive/PlantVillage_Model"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "plant_disease_mobilenetv2.keras")
CLASS_NAMES_PATH = os.path.join(MODEL_DIR, "class_names.json")

# ============================================================
# 2. SETTINGS
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 38
SEED = 42

# ============================================================
# 3. LOAD DATASETS
# ============================================================

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

# ============================================================
# 4. CLASS NAMES
# ============================================================

class_names = train_ds.class_names

print("\nNumber of classes:", len(class_names))

for i, name in enumerate(class_names):
    print(i, ":", name)

# Save class names
with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(class_names, f, indent=4)

print("\nClass names saved ✅")
print(CLASS_NAMES_PATH)

# ============================================================
# 5. PERFORMANCE OPTIMIZATION
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# ============================================================
# 6. DATA AUGMENTATION
# ============================================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

# ============================================================
# 7. MOBILE NET V2
# ============================================================

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False

# ============================================================
# 8. BUILD MODEL
# ============================================================

inputs = layers.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# MobileNetV2 preprocessing
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

x = layers.Dense(128, activation="relu")(x)

x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = models.Model(
    inputs,
    outputs
)

# ============================================================
# 9. COMPILE
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 10. MODEL SUMMARY
# ============================================================

model.summary()

# ============================================================
# 11. CALLBACKS
# ============================================================

callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

# ============================================================
# 12. TRAIN
# ============================================================

EPOCHS = 10

print("\n====================================")
print("STARTING TRAINING")
print("====================================\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("\n====================================")
print("TRAINING COMPLETED ✅")
print("====================================")

print("\nModel saved at:")
print(MODEL_PATH)

Found 12130 files belonging to 38 classes.
Found 2596 files belonging to 38 classes.
Found 2608 files belonging to 38 classes.

Number of classes: 38
0 : Apple___Apple_scab
1 : Apple___Black_rot
2 : Apple___Cedar_apple_rust
3 : Apple___healthy
4 : Blueberry___healthy
5 : Cherry_(including_sour)___Powdery_mildew
6 : Cherry_(including_sour)___healthy
7 : Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
8 : Corn_(maize)___Common_rust_
9 : Corn_(maize)___Northern_Leaf_Blight
10 : Corn_(maize)___healthy
11 : Grape___Black_rot
12 : Grape___Esca_(Black_Measles)
13 : Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14 : Grape___healthy
15 : Orange___Haunglongbing_(Citrus_greening)
16 : Peach___Bacterial_spot
17 : Peach___healthy
18 : Pepper,_bell___Bacterial_spot
19 : Pepper,_bell___healthy
20 : Potato___Early_blight
21 : Potato___Late_blight
22 : Potato___healthy
23 : Raspberry___healthy
24 : Soybean___healthy
25 : Squash___Powdery_mildew
26 : Strawberry___Leaf_scorch
27 : Strawberry___healthy
28

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,854 (9.26 MB)

 Trainable params: 168,870 (659.65 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


STARTING TRAINING

Epoch 1/10
380/380 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.4324 - loss: 2.1259
Epoch 1: val_accuracy improved from None to 0.87558, saving model to /content/drive/MyDrive/PlantVillage_Model/plant_disease_mobilenetv2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/PlantVillage_Model/plant_disease_mobilenetv2.keras
380/380 ━━━━━━━━━━━━━━━━━━━━ 57s 118ms/step - accuracy: 0.6133 - loss: 1.3690 - val_accuracy: 0.8756 - val_loss: 0.4479 - learning_rate: 0.0010
Epoch 2/10
379/380 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.7977 - loss: 0.6537
Epoch 2: val_accuracy improved from 0.87558 to 0.89484, saving model to /content/drive/MyDrive/PlantVillage_Model/plant_disease_mobilenetv2.keras

Epoch 2: finished saving model to /content/drive/MyDrive/PlantVillage_Model/plant_disease_mobilenetv2.keras
380/380 ━━━━━━━━━━━━━━━━━━━━ 42s 112ms/step - accuracy: 0.8080 - loss: 0.6113 - val_accuracy: 0.8948 - val_loss: 0.3546 - learning_rate: 0.0010
Epoch 3/10


In [17]:
# ============================================
# FINAL TEST EVALUATION
# ============================================

import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/PlantVillage_Model/plant_disease_mobilenetv2.keras"
TEST_DIR = "/content/drive/MyDrive/PlantVillage_Split/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load best saved model
model = tf.keras.models.load_model(MODEL_PATH)

# Load test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

# Evaluate
test_loss, test_accuracy = model.evaluate(test_ds)

print("\n====================================")
print("FINAL TEST RESULTS")
print("====================================")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print("====================================")

Found 2608 files belonging to 38 classes.
82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 75ms/step - accuracy: 0.9314 - loss: 0.2122

FINAL TEST RESULTS
Test Loss     : 0.2122
Test Accuracy : 0.9314
Test Accuracy : 93.14%
